# Metadata Demo Files
Goal is to add flat ACDD variables in a different netCDF group and tie them together with ancillary variables
Going to start with the following two cruises
* 2013 P02
  * 2 legs put together
  * Different chisci
  * CFC group completly swapped in the middle (different PIs)
* 2021 A20
  * Some minor overlap in PI
  * Different ship
  * Different program (GO-SHIP vs CLIVAR)
  * Different basin

The idea is to tie variables with the same metadata together using a grouping variable in the different netCDF group

e.g. in the root group `/silicate` variable, the `ancillary_variables` attribute might contain `/meta/group_var0` which itself references the other actual metadata variables. The meta variables MUST share dims and the coordinates (EXPOCODE, etc..).

This will make use of the somewhat new DataTree functions in xarray

In [1]:
# only load data if needed
import requests
def download_to_file(url):
    r = requests.get(url)
    fname = r.headers.get("Content-Disposition").split("''")[1]
    with open(fname, "wb") as fo:
        fo.write(r.content)
for url in [
    "https://cchdo.ucsd.edu/data/38748/318M20130321_bottle.nc",
    "https://cchdo.ucsd.edu/data/39457/318M20130321_ctd.nc",
    "https://cchdo.ucsd.edu/data/41458/325020210316_bottle.nc",
    "https://cchdo.ucsd.edu/data/24830/325020210316_do.pdf",
    "https://cchdo.ucsd.edu/data/3928/p02_318M20130321do.pdf"
           ]:
    download_to_file(url)

In [34]:
import xarray as xr
import numpy as np
from uuid import uuid4
from cchdo.hydro import accessors

In [3]:
p02 = xr.load_datatree("318M20130321_bottle.nc")
a20 = xr.load_datatree("325020210316_bottle.nc")

In [4]:
# We need to separate each leg of p02 for this test
p02w = p02["/section_id"] == "P02W"
p02e = p02["/section_id"] == "P02E"

In [5]:
encoding = {"dtype": "S1", "zlib": True, "complevel": 9}
# lets build the metadata creator for tracers, the complex case...
tracer_vars = ["cfc_11", "cfc_12", "cfc_113", "sulfur_hexifluoride"]

# leg1
p02w_creator_name = "David Ho"
p02w_creator_role = "Principal Investigator" # make this a controlled list
p02w_creator_email = "ho@hawaii.edu"
p02w_creator_institution = "UHawaii" # from the cruise report... how to include ror? not in ACDD so omitting for now
p02w_processing_level = "final" # make this a controlled list
# there are no cfc/sf6 specific comments

p02e_creator_name = "Dong-Ha Min"
p02e_creator_role = "Principal Investigator"
p02e_creator_email = "dongha@austin.utexas.edu"
p02e_creator_institution = "UT-Austin"
p02e_processing_level = "final"

p02_creator_name = xr.DataArray(np.empty(p02["/expocode"].shape, dtype=object), dims=("N_PROF"), attrs={"acdd_role": "creator_name"}, name=f"meta_{uuid4()}")
p02_creator_name[p02w] = p02w_creator_name
p02_creator_name[p02e] = p02e_creator_name
p02_creator_name.encoding = encoding

p02_creator_role = xr.DataArray(np.empty(p02["/expocode"].shape, dtype=object), dims=("N_PROF"), attrs={"acdd_role": "creator_role"}, name=f"meta_{uuid4()}")
p02_creator_role[p02w] = p02w_creator_role
p02_creator_role[p02e] = p02e_creator_role
p02_creator_role.encoding = encoding

p02_creator_email = xr.DataArray(np.empty(p02["/expocode"].shape, dtype=object), dims=("N_PROF"), attrs={"acdd_role": "creator_email"}, name=f"meta_{uuid4()}")
p02_creator_email[p02w] = p02w_creator_email
p02_creator_email[p02e] = p02e_creator_email
p02_creator_email.encoding = encoding

p02_creator_institution = xr.DataArray(np.empty(p02["/expocode"].shape, dtype=object), dims=("N_PROF"), attrs={"acdd_role": "creator_institution"}, name=f"meta_{uuid4()}")
p02_creator_institution[p02w] = p02w_creator_institution
p02_creator_institution[p02e] = p02e_creator_institution
p02_creator_institution.encoding = encoding

p02_processing_level = xr.DataArray(np.empty(p02["/expocode"].shape, dtype=object), dims=("N_PROF"), attrs={"acdd_role": "processing_level"}, name=f"meta_{uuid4()}")
p02_processing_level[p02w] = p02w_processing_level
p02_processing_level[p02e] = p02e_processing_level
p02_processing_level.encoding = encoding

creator_vars = [p02_creator_name, p02_creator_role, p02_creator_email, p02_creator_institution, p02_processing_level]
grouping_var = xr.DataArray(name=f"group_{uuid4()}", attrs={"meta_vars": [v.name for v in creator_vars]})

# simple case

def add_creator_processing_level_scalar(dt, var_list, creator_name, creator_role, creator_email, creator_institution, processing_level=None):
    # warning... side effect function
    cn_da = xr.DataArray(creator_name, attrs={"acdd_role": "creator_name"}, name=f"meta_{uuid4()}")
    cn_da.encoding = encoding

    cr_da = xr.DataArray(creator_role, attrs={"acdd_role": "creator_role"}, name=f"meta_{uuid4()}")
    cr_da.encoding = encoding
    
    ce_da = xr.DataArray(creator_email, attrs={"acdd_role": "creator_email"}, name=f"meta_{uuid4()}")
    ce_da.encoding = encoding

    ci_da = xr.DataArray(creator_institution, attrs={"acdd_role": "creator_institution"}, name=f"meta_{uuid4()}")
    ci_da.encoding = encoding
    
    vars_to_add = [cn_da, cr_da, ce_da, ci_da]
    if processing_level is not None:
        pl_da = xr.DataArray(processing_level, attrs={"acdd_role": "processing_level"}, name=f"meta_{uuid4()}")
        pl_da.encoding = encoding
        vars_to_add.append(pl_da)
        
    grouping_var = xr.DataArray(name=f"group_{uuid4()}", attrs={"meta_vars": [v.name for v in vars_to_add]})

    try:
        dt["/meta"]
    except KeyError:
        dt["/meta"] = xr.DataSet()

    for var in [*vars_to_add, grouping_var]:
        dt[f"/meta/{var.name}"] = var

    for var in var_list:
        if "ancillary_variables" in dt[var].attrs:
            current = dt[var].attrs["ancillary_variables"]
            dt[var].attrs["ancillary_variables"] = f"{current} /meta/{grouping_var.name}"
        else:
            dt[var].attrs["ancillary_variables"] = f"/meta/{grouping_var.name}"

p02_out = p02.copy()

# just manually assign stuff for now... ?
meta = xr.Dataset(
    {v.name: v for v in [*creator_vars, grouping_var]}
)
p02_out["/meta"] = meta
for v in tracer_vars:
    
    if "ancillary_variables" in p02_out[v].attrs:
        current = p02_out[v].attrs["ancillary_variables"]
        p02_out[v].attrs["ancillary_variables"] = f"{current} /meta/{grouping_var.name}"
    else:
        p02_out[v].attrs["ancillary_variables"] = f"/meta/{grouping_var.name}"

add_creator_processing_level_scalar(p02_out, ["bottle_number", "ctd_temperature", "ctd_salinity", "bottle_salinity", "ctd_oxygen", "oxygen", "silicate", "nitrate", "nitrite", "phosphate", "ref_temperature"], "James H. Swift", "Principal Investigator", "jswift@ucsd.edu", "UCSD/SIO", "final")
add_creator_processing_level_scalar(p02_out, ["tritium", "helium", "delta_helium_3"], "William Jenkins", "Principal Investigator", "wjenkins@whoi.edu", "WHOI", "final")
add_creator_processing_level_scalar(p02_out, ["total_carbon"], "Richard Feely", "Principal Investigator", "Richard.A.Feely@noaa.gov", "NOAA/PMEL", "final")
add_creator_processing_level_scalar(p02_out, ["total_alkalinity", "ph_total_h_scale"], "Andrew Dickson", "Principal Investigator", "adickson@ucsd.edu", "UCSD/SIO", "final")

p02_out.to_netcdf("p02_out.nc")

In [6]:
# demo what is possible?
def resolve_ancillary(dt, var):
    da = dt[var]
    resolved = {}
    if (ans := da.attrs.get("ancillary_variables")) is not None:
        for var in ans.split():
            resolved[var] = dt[var]
    return resolved
def resolve_meta(dt, var):
    da = dt[var]
    resolved = {}
    if (ans := da.attrs.get("meta_vars")) is not None:
        for var in ans:
            resolved[f"/meta/{var}"] = dt[f"/meta/{var}"]
    return resolved
    
def filter_by_meta(dt, **kwargs):
    # goal will be to collect variables that need to be included in the output
    # only things with ancillary matter
    filtered_vars = []
    has_ans = dt.ds.filter_by_attrs(ancillary_variables=lambda v: v is not None)
    for var in has_ans.variables: # we will want this to just recurse somehow probably...
        ans = resolve_ancillary(dt, var)
        for varN in ans:
            meta = resolve_meta(dt, varN)
            for varM in meta.values():
                for key, value in kwargs.items():
                    if "acdd_role" not in varM.attrs:
                        continue
                    if varM.attrs["acdd_role"] != key:
                        continue
                    if np.any(varM == value).item():
                        filtered_vars.append(var)
                        continue
    return dt.ds[filtered_vars]

# all the vars where jim is the PI
filter_by_meta(p02_out, creator_name="James H. Swift")

<xarray.Dataset> Size: 602kB
Dimensions:          (N_PROF: 159, N_LEVELS: 36)
Coordinates:
    expocode         (N_PROF) object 1kB '318M20130321' ... '318M20130321'
    station          (N_PROF) object 1kB '1' '2' '3' '4' ... '157' '158' '159'
    cast             (N_PROF) int32 636B 2 1 1 1 1 1 1 1 1 ... 1 1 1 1 1 2 1 1 1
    time             (N_PROF) datetime64[ns] 1kB 2013-03-22T22:05:00.00000025...
    latitude         (N_PROF) float64 1kB 32.51 32.45 32.41 ... 32.64 32.64
    longitude        (N_PROF) float64 1kB 133.0 133.2 133.3 ... -117.5 -117.4
    sample           (N_PROF, N_LEVELS) object 46kB '9' '8' '10' ... '' '' ''
    pressure         (N_PROF, N_LEVELS) float64 46kB 8.3 8.3 8.4 ... nan nan nan
Dimensions without coordinates: N_PROF, N_LEVELS
Data variables:
    bottle_number    (N_PROF, N_LEVELS) object 46kB '9' '8' '10' ... '' '' ''
    ctd_temperature  (N_PROF, N_LEVELS) float64 46kB 19.21 19.19 ... nan nan
    ctd_salinity     (N_PROF, N_LEVELS) float64 46kB 34.69 34.69 ... nan nan
    bottle_salinity  (N_PROF, N_LEVELS) float64 46kB nan 34.69 nan ... nan nan
    ctd_oxygen       (N_PROF, N_LEVELS) float64 46kB 220.5 220.6 ... nan nan
    oxygen           (N_PROF, N_LEVELS) float64 46kB nan 220.7 nan ... nan nan
    silicate         (N_PROF, N_LEVELS) float64 46kB nan 3.51 nan ... nan nan
    nitrate          (N_PROF, N_LEVELS) float64 46kB nan 1.55 nan ... nan nan
    nitrite          (N_PROF, N_LEVELS) float64 46kB nan 0.19 nan ... nan nan
    phosphate        (N_PROF, N_LEVELS) float64 46kB nan 0.19 nan ... nan nan
    ref_temperature  (N_PROF, N_LEVELS) float64 46kB 19.21 19.19 ... nan nan
Attributes:
    Conventions:               CF-1.8 CCHDO-1.0
    cchdo_software_version:    hydro 1.0.2.3
    cchdo_parameters_version:  params 0.1.21
    comments:                  BOTTLE,20220817CCHSIOAMB\n Changed I-129 units...
    featureType:               profile

In [7]:
#includes Jim and Dickson
filter_by_meta(p02_out, creator_institution="UCSD/SIO")

<xarray.Dataset> Size: 694kB
Dimensions:           (N_PROF: 159, N_LEVELS: 36)
Coordinates:
    expocode          (N_PROF) object 1kB '318M20130321' ... '318M20130321'
    station           (N_PROF) object 1kB '1' '2' '3' '4' ... '157' '158' '159'
    cast              (N_PROF) int32 636B 2 1 1 1 1 1 1 1 1 ... 1 1 1 1 2 1 1 1
    time              (N_PROF) datetime64[ns] 1kB 2013-03-22T22:05:00.0000002...
    latitude          (N_PROF) float64 1kB 32.51 32.45 32.41 ... 32.64 32.64
    longitude         (N_PROF) float64 1kB 133.0 133.2 133.3 ... -117.5 -117.4
    sample            (N_PROF, N_LEVELS) object 46kB '9' '8' '10' ... '' '' ''
    pressure          (N_PROF, N_LEVELS) float64 46kB 8.3 8.3 8.4 ... nan nan
Dimensions without coordinates: N_PROF, N_LEVELS
Data variables: (12/13)
    bottle_number     (N_PROF, N_LEVELS) object 46kB '9' '8' '10' ... '' '' ''
    ctd_temperature   (N_PROF, N_LEVELS) float64 46kB 19.21 19.19 ... nan nan
    ctd_salinity      (N_PROF, N_LEVELS) float64 46kB 34.69 34.69 ... nan nan
    bottle_salinity   (N_PROF, N_LEVELS) float64 46kB nan 34.69 nan ... nan nan
    ctd_oxygen        (N_PROF, N_LEVELS) float64 46kB 220.5 220.6 ... nan nan
    oxygen            (N_PROF, N_LEVELS) float64 46kB nan 220.7 nan ... nan nan
    ...                ...
    nitrate           (N_PROF, N_LEVELS) float64 46kB nan 1.55 nan ... nan nan
    nitrite           (N_PROF, N_LEVELS) float64 46kB nan 0.19 nan ... nan nan
    phosphate         (N_PROF, N_LEVELS) float64 46kB nan 0.19 nan ... nan nan
    total_alkalinity  (N_PROF, N_LEVELS) float64 46kB nan 2.28e+03 ... nan nan
    ph_total_h_scale  (N_PROF, N_LEVELS) float64 46kB nan 8.008 nan ... nan nan
    ref_temperature   (N_PROF, N_LEVELS) float64 46kB 19.21 19.19 ... nan nan
Attributes:
    Conventions:               CF-1.8 CCHDO-1.0
    cchdo_software_version:    hydro 1.0.2.3
    cchdo_parameters_version:  params 0.1.21
    comments:                  BOTTLE,20220817CCHSIOAMB\n Changed I-129 units...
    featureType:               profile

In [33]:
# lets try adding the URL to data in the data to see what happens in ODV, the ODV docs say it _should_ present as a link to click on
# this is also the POC url...
hplc_url = "https://seabass.gsfc.nasa.gov/cruise/A20_2021"

# the following is "cheating" and should probably not be done in production this way...
a20_out = a20.copy()
# lets only add data where there is flag 1s
sampled = a20_out.hplc_placeholder_qc==1
a20_out.hplc_placeholder.values[sampled.values] = hplc_url

# the warning about string dims is expected
a20_out.to_netcdf("325020210316_bottle_with_hplc.nc")

/var/folders/y1/63dlf4614h5d2cgr5g1t_5lh0000gn/T/ipykernel_59658/3300128417.py:12: UserWarning: String dimension naming mismatch on variable None. 'string1' provided by encoding, but data has length of '45'. Using 'string45' instead of 'string1' to prevent possible naming clash.
To silence this warning either remove 'char_dim_name' from encoding or provide a fitting name.
  a20_out.to_netcdf("325020210316_bottle_with_hplc.nc")


In [37]:
# in testing ODV... opening the nc file didn't let me filter on the HPLC var
# opening the netCDF and saving+reopening the collection allowed the filtering based on HPLC availability
# the exchange version caused ODV to think data was available everywhere due to interpreting the string "-999" as a value (somewhat correctly IMO)
# however, we can filter on wildcards, so filtering the HPLC var on "http*" showed all the locations, it was... really neat
a20_out.ds.cchdo.to_exchange("325020210316_bottle_with_hplc_hy1.csv")